## Setup

In [1]:
%load_ext autoreload
%autoreload 2
import math
import torch
import numpy as np
import mediapy
import matplotlib.pyplot as plt
from torch.nn.functional import interpolate

from hydra import initialize, compose
from omegaconf import OmegaConf

from dreamerv4uwm.models.utils import load_tokenizer, load_denoiser
from dreamerv4uwm.datasets import G1ChunkDataset, ShardedHDF5Dataset
from dreamerv4uwm.sampling_new import (
    make_is_horizon,
    ray_sampler,
    grid_loss_probe,
)

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
resolution = (256, 256)

In [14]:
# Config
# with initialize(version_base=None, config_path='../scripts/config'):
#     cfg = compose(config_name='dynamics/pushT')
with initialize(version_base=None, config_path='../scripts/config'):
    cfg = compose(
        config_name='dynamics/pushT-large',
        overrides=['denoiser.horizon_aware=false'],
    )

print('horizon_aware:', cfg.denoiser.get('horizon_aware', False))
print('train.unified.mode_probs:', OmegaConf.to_container(cfg.train.unified.mode_probs))

horizon_aware: False
train.unified.mode_probs: {'unified': 0.333, 'video_pretraining': 0.333, 'action_pretraining': 0.333}


/home/mim-server/miniconda3/envs/dreamerv4uwm/lib/python3.11/site-packages/hydra/_internal/defaults_list.py:251: UserWarning: In 'dynamics/pushT-large': Defaults list is missing `_self_`. See https://hydra.cc/docs/1.2/upgrades/1.0_to_1.1/default_composition_order for more information
  warnings.warn(msg, UserWarning)


In [15]:
# Point these at the new-pipeline checkpoint and the matching tokenizer.
DYNAMICS_CKPT = '/home/mim-server/projects/rooholla/dreamerV4-UWM/checkpoints/blockcausal/pushT-post-train/97500.pt'
TOKENIZER_CKPT = '/home/mim-server/projects/rooholla/dreamerV4-UWM/checkpoints/tokenizer/pushT.pt'
cfg.dynamics_ckpt = DYNAMICS_CKPT
cfg.tokenizer_ckpt = TOKENIZER_CKPT

denoiser = load_denoiser(cfg, device, max_num_forward_steps=300).eval().cuda()
tokenizer = load_tokenizer(cfg, device, max_num_forward_steps=300).eval().cuda()
print(denoiser.model.frame_id_embedder)  # should be DiscreteEmbedder(2, model_dim) when horizon_aware=true

None


In [16]:
# A single held-out window for diagnostics.
import mediapy
from torch.nn.functional import interpolate

DATA_PATH = "/home/mim-server/datasets/pushT/h5/play"
dataset = ShardedHDF5Dataset(
        data_dir=DATA_PATH,
        window_size=64,
        stride=1,
        split='train',
        train_fraction=0.9,
        split_seed=123,
    )

# Arbitrary index into that episode
batch = dataset[torch.randint(len(dataset), (1,)).item()]
# imgs = batch["image"][:,[2, 1, 0], :, :]  # (T, C, H, W)
imgs = batch["image"]  # (T, C, H, W)
actions = batch["action"][:,:cfg.denoiser.n_actions][None].cuda()  # (1, T, N_lat, D_lat)
imgs = interpolate(imgs, resolution).to(device=device)[None] # resize to tokenizer resolution

with torch.no_grad():
    with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
        latents = tokenizer.encode(imgs)
        imgs_recon = tokenizer.decode(latents)

print('imgs:', imgs.shape, 'actions:', actions.shape, 'latents:', latents.shape)

Train split: 64694 windows from 11 episodes
imgs: torch.Size([1, 64, 3, 256, 256]) actions: torch.Size([1, 64, 2]) latents: torch.Size([1, 64, 256, 32])


In [17]:
# Display helpers (mirrors g1-sampling.ipynb).
def plotVideo(video, fps=10):
    arr = (video.cpu().permute(0, 2, 3, 1).to(torch.float32).numpy() * 255).clip(0, 255).astype(np.uint8)
    mediapy.show_video(arr, fps=fps)

def plotSnapshots(video, n_frames=5, border_width=2, figsize=None):
    T = video.shape[0]
    idx = np.linspace(0, T - 1, n_frames, dtype=int)
    frames = (video[idx].cpu().permute(0, 2, 3, 1).to(torch.float32).numpy() * 255).clip(0, 255).astype(np.uint8)
    H, W, C = frames.shape[1:]
    border = np.zeros((H, border_width, C), dtype=np.uint8)
    parts = []
    for i, f in enumerate(frames):
        if i > 0: parts.append(border)
        parts.append(f)
    strip = np.concatenate(parts, axis=1)
    figsize = figsize or (n_frames * 3, 3)
    fig, ax = plt.subplots(1, 1, figsize=figsize)
    ax.imshow(strip); ax.axis('off'); plt.tight_layout(); plt.show()

def plotActions(actions, dim_labels=None, ylim=(-1.0, 1.0)):
    act = actions.cpu().float().numpy() if hasattr(actions, 'cpu') else np.array(actions)
    fig, ax = plt.subplots(1, 1, figsize=(8, 3))
    ax.axhline(0, color='black', linestyle='--', linewidth=1)
    n_dim = act.shape[-1]
    for d in range(n_dim):
        label = dim_labels[d] if dim_labels else f'dim {d}'
        ax.plot(act[:, d], linewidth=1.2, alpha=0.85, label=label)
    ax.set_ylim(*ylim)
    if n_dim <= 6:
        ax.legend(fontsize=9, loc='upper right')
    plt.tight_layout(); plt.show()

## Split into context / horizon

In [18]:
T_ctx = 8
T_hor = 32
ctx_latents = latents[:, :T_ctx].clone()
ctx_actions = actions[:, :T_ctx].clone()
hor_latents = latents[:, T_ctx:T_ctx + T_hor].clone()
hor_actions = actions[:, T_ctx:T_ctx + T_hor].clone()
print('ctx:', ctx_latents.shape, ctx_actions.shape, 'hor:', hor_latents.shape, hor_actions.shape)

ctx: torch.Size([1, 8, 256, 32]) torch.Size([1, 8, 2]) hor: torch.Size([1, 32, 256, 32]) torch.Size([1, 32, 2])


In [19]:
ctx_latents.shape

torch.Size([1, 8, 256, 32])

In [20]:
from typing import Optional, Tuple

def _quantize_tau_to_idx(tau: float, N: int) -> int:
    """Map continuous τ ∈ [0, 1] to a valid embedding index in [0, N-1]."""
    return max(0, min(N - 1, int(round(tau * N))))


@torch.no_grad()
def videoSampler(
    denoiser,
    num_diffusion_steps: int,
    video_length: int=16,
    device='cuda:0'
) -> Tuple[torch.Tensor, torch.Tensor]:
    N = denoiser.cfg.denoiser.num_noise_levels
    is_horizon = make_is_horizon(video_length, ctx_len=0, device=device, all_bidir=True)


    # Horizon state: partial-noise mix per frame.
    z = torch.randn(1, video_length, 256, 32, device=device, dtype=torch.float32)
    a = torch.randn(1, video_length, 2, device=device, dtype=torch.float32)

    
    K = int(num_diffusion_steps)
    step_idx_t   = torch.zeros((1, video_length), dtype=torch.long, device=device)    
    act_tau_cond_idx = _quantize_tau_to_idx(0.0, N) 
    obs_tau_cond_idx = _quantize_tau_to_idx(0.0, N) 
    obs_sigma_idx = torch.full((1, video_length), obs_tau_cond_idx, dtype=torch.long, device=device)
    act_sigma_idx = torch.full((1, video_length), act_tau_cond_idx, dtype=torch.long, device=device)
    dt_state_b = 1./num_diffusion_steps
    for k in range(K):
        current_tau = float(k) / float(K)
        current_tau_idx = _quantize_tau_to_idx(current_tau, N)
        obs_sigma_idx[:] = current_tau_idx

        z_hat, _, _ = denoiser(
            noisy_act=a,
            noisy_obs=z,
            obs_sigma_idx=obs_sigma_idx,
            obs_step_idx=step_idx_t,
            act_sigma_idx=act_sigma_idx,
            act_step_idx=step_idx_t,
            is_horizon=is_horizon,
        )
        v = (z_hat - z) / (1. - current_tau)
        z[:] = z[:] + v * dt_state_b

    return z_hat

In [22]:
z = videoSampler(denoiser, 16, 8)
with torch.no_grad():
    img_vid = tokenizer.decode(z)

plotVideo(img_vid[0].to(torch.float32))

This browser does not support the video tag.


In [23]:
@torch.no_grad()
def getVelFiled(
    denoiser,
    z, 
    tau,
    context = None,
    context_tau = 0.9,
    device='cuda:0',
    context_at_end = False,
) -> Tuple[torch.Tensor, torch.Tensor]:
    
    if context is None:
        z_t = z
    else:
        if not context_at_end:
            z_t = torch.concat([context, z], dim=1)
        else:
            z_t = torch.concat([z, context], dim=1)
    T = z_t.shape[1]

    N = denoiser.cfg.denoiser.num_noise_levels
    is_horizon = make_is_horizon(T, ctx_len=0, device=device, all_bidir=True)
    a = torch.randn(1, T, 2, device=device, dtype=torch.float32)
    step_idx_t   = torch.zeros((1, T), dtype=torch.long, device=device)    
    act_tau_cond_idx = _quantize_tau_to_idx(0.0, N) 
    obs_tau_cond_idx = _quantize_tau_to_idx(tau, N) 
    obs_sigma_idx = torch.full((1, T), obs_tau_cond_idx, dtype=torch.long, device=device)
    act_sigma_idx = torch.full((1, T), act_tau_cond_idx, dtype=torch.long, device=device)

    if context is not None:
        T_ctx=context.shape[1]
        ctx_tau_idx = _quantize_tau_to_idx(context_tau, N)
        obs_sigma_idx[:, :T_ctx] = ctx_tau_idx

    z_hat, _, _ = denoiser(
        noisy_act=a,
        noisy_obs=z_t,
        obs_sigma_idx=obs_sigma_idx,
        obs_step_idx=step_idx_t,
        act_sigma_idx=act_sigma_idx,
        act_step_idx=step_idx_t,
        is_horizon=is_horizon,
    )
    v = (z_hat - z_t) / (1. - tau)
    x = z_hat
    if context is not None:
        if not context_at_end:
            return x[:, T_ctx:], v[:, T_ctx:]
        else:
            return x[:, :-T_ctx], v[:, :-T_ctx]
    else:
        return x, v

In [29]:
video_length=4
n_steps = 32
dt = 1./float(n_steps)
z_t = torch.randn(1, video_length, 256, 32, device=device, dtype=torch.float32)
context = latents[:, 10, ...].clone().unsqueeze(1)  # (1, 1, 256, 32)


for k in range(n_steps):
    tau = k/n_steps
    x, v = getVelFiled(denoiser, z_t, tau, context=context, context_tau=0.8, context_at_end=True)
    z_t = z_t + v*dt


with torch.no_grad():
    img_vid = tokenizer.decode(x)
    context_vid = tokenizer.decode(context)

plotVideo(img_vid[0].to(torch.float32))
plotVideo(context_vid[0].to(torch.float32))


In [28]:
z_t.shape

torch.Size([1, 4, 256, 32])

In [30]:
video_length=8
n_steps = 8

w = 0.7
dt = 1./float(n_steps)
z_t = torch.randn(1, video_length, 256, 32, device=device, dtype=torch.float32)
context = latents[:, 0, ...].clone().unsqueeze(1)  # (1, 1, 256, 32)

for k in range(n_steps):
    tau = k/n_steps
    x_uncon, v_uncon = getVelFiled(denoiser, z_t, tau)
    x_con, v_con = getVelFiled(denoiser, z_t, tau, context=context, context_tau=1.0, context_at_end=False)
    
    v = v_uncon + w*(v_con-v_uncon)
    z_t[:] = z_t + v*dt


with torch.no_grad():
    img_vid = tokenizer.decode(x_uncon)
    img_vid_con = tokenizer.decode(context)

plotVideo(img_vid_con[0].to(torch.float32))
plotVideo(img_vid[0].to(torch.float32))


In [31]:
vid = torch.concat()

TypeError: concat() received an invalid combination of arguments - got (), but expected one of:
 * (tuple of Tensors tensors, int dim = 0, *, Tensor out = None)
 * (tuple of Tensors tensors, name dim, *, Tensor out = None)
